<a href="https://colab.research.google.com/github/diaconescualexandra/ASR_TTS_NKUA/blob/main/notebooks/01_asr_librispeech_citrinet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip uninstall -y numpy pandas scipy scikit-learn librosa numba nemo-toolkit
!pip install -q numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.3.2 librosa==0.10.1 numba==0.59.1
!pip install -q "nemo-toolkit[asr,tts]==2.5.0"

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: librosa 0.11.0
Uninstalling librosa-0.11.0:
  Successfully uninstalled librosa-0.11.0
Found existing installation: numba 0.60.0
Uninstalling numba-0.60.0:
  Successfully uninstalled numba-0.60.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 83.2 MB/s eta 0:00:00
   ━━━━

In [1]:
import numpy as np
import pandas as pd
import torch
import nemo.collections.asr as nemo_asr

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print("NeMo ASR works")

[NeMo W 2026-05-28 10:11:33 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
      m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
    
[NeMo W 2026-05-28 10:11:33 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
      m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
    
[NeMo W 2026-05-28 10:11:33 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(flt)p?( \(default\))?$', token):
    
[NeMo W 2026-05-28 10:11:33 nemo_logging:405] /usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
      elif re.match('(dbl)p?( \(default\))?$', token):
    


numpy: 1.26.4
pandas: 2.2.2
cuda: True
Tesla T4
NeMo ASR works


In [3]:
# creating the ASR dataset
!mkdir -p data/librispeech
%cd /content
!wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz
!tar -xzf dev-clean.tar.gz -C /content/data/librispeech

/content


In [4]:
!find /content/data/librispeech/LibriSpeech/dev-clean -name "*.flac" | head

/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0003.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0012.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0010.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0013.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0007.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0016.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0019.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0000.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0009.flac
/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0018.flac


In [6]:
# cretae manifest files for NeMo

import os
import json
import random
import soundfile as sf
from pathlib import Path

librispeech_root = Path("/content/data/librispeech/LibriSpeech/dev-clean")
output_dir = Path("/content/manifests")
output_dir.mkdir(parents=True, exist_ok=True)

items = []

for trans_file in librispeech_root.rglob("*.trans.txt"):
    with open(trans_file, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            utt_id = parts[0]
            text = parts[1].lower()

            audio_path = trans_file.parent / f"{utt_id}.flac"
            info = sf.info(str(audio_path))
            duration = info.duration

            items.append({
                "audio_filepath": str(audio_path),
                "duration": duration,
                "text": text
            })

print("Total samples:", len(items))
print(items[0])

#split dataset intro train test validation sets
random.seed(42)
random.shuffle(items)

n = len(items)
train = items[:int(0.8*n)]
val = items[int(0.8*n):int(0.9*n)]
test = items[int(0.9*n):]

def write_manifest(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

write_manifest(train, output_dir / "train_manifest.json")
write_manifest(val, output_dir / "val_manifest.json")
write_manifest(test, output_dir / "test_manifest.json")

print(len(train), len(val), len(test))

Total samples: 2703
{'audio_filepath': '/content/data/librispeech/LibriSpeech/dev-clean/2902/9006/2902-9006-0000.flac', 'duration': 4.8, 'text': 'one who writes of such an era labours under a troublesome disadvantage'}
2162 270 271


In [8]:
# check files
!head -n 2 /content/manifests/train_manifest.json

{"audio_filepath": "/content/data/librispeech/LibriSpeech/dev-clean/5536/43358/5536-43358-0013.flac", "duration": 10.4, "text": "from the sun as the universal father proceeds the quickening principle in nature and in the patient and fruitful womb of our mother the earth are hidden embryos of plants and men"}
{"audio_filepath": "/content/data/librispeech/LibriSpeech/dev-clean/5694/64038/5694-64038-0000.flac", "duration": 2.595, "text": "advance into tennessee"}


In [11]:
# baseline inference

import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(
    model_name="stt_en_citrinet_256"
)

# testing the baseline inference on 5 files
sample_audio = [item["audio_filepath"] for item in test[:5]]
sample_refs = [item["text"] for item in test[:5]]

preds = asr_model.transcribe(sample_audio)

for ref, pred in zip(sample_refs, preds):
    print("REF :", ref)
    print("PRED:", pred)
    print("-" * 80)

[NeMo I 2026-05-28 10:17:10 nemo_logging:393] Found existing object /root/.cache/torch/NeMo/NeMo_2.5.0/stt_en_citrinet_256/91a9cc5850784b2065e8a0aa3d526fd9/stt_en_citrinet_256.nemo.
[NeMo I 2026-05-28 10:17:10 nemo_logging:393] Re-using file from: /root/.cache/torch/NeMo/NeMo_2.5.0/stt_en_citrinet_256/91a9cc5850784b2065e8a0aa3d526fd9/stt_en_citrinet_256.nemo
[NeMo I 2026-05-28 10:17:10 nemo_logging:393] Instantiating model from pre-trained checkpoint
[NeMo I 2026-05-28 10:17:11 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-05-28 10:17:11 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    trim_silence: true
    max_duration: 16.7
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    use_start_end_token: false
    
[NeMo W 2026-05-28 10:17:11 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: null
    sample_rate: 16000
    batch_size: 32
    shuffle: false
    use_start_end_token: false
    
[NeMo W 2026-05-28 10:17:11 nemo_logging:405] Please call the ModelPT.setup_test_data() or ModelPT.setup_multiple_test_data() method a

[NeMo I 2026-05-28 10:17:11 nemo_logging:393] PADDING: 16
[NeMo I 2026-05-28 10:17:12 nemo_logging:393] Model EncDecCTCModelBPE was successfully restored from /root/.cache/torch/NeMo/NeMo_2.5.0/stt_en_citrinet_256/91a9cc5850784b2065e8a0aa3d526fd9/stt_en_citrinet_256.nemo.


Transcribing: 100%|██████████| 2/2 [00:01<00:00,  1.13it/s]

REF : add two tablespoons thick sour cream two tablespoons sugar a sprinkle of mustard and half cup of vinegar
PRED: Hypothesis(score=tensor(-1.2399), y_sequence=tensor([1024, 1024, 1024, 1024,  472, 1024,  211, 1024,   74, 1024,  264, 1024,
           1,  331,  111,  111,    1, 1024,  283,  366, 1024, 1024,   39, 1024,
         109, 1024,   63,   24,  362, 1024, 1024, 1024, 1024,  211, 1024, 1024,
          74, 1024,  264, 1024,    1,  331,  111,  111,    1, 1024, 1024,  204,
          38,   58, 1024, 1024, 1024, 1024,    4, 1024,  222,   84,   18,   86,
          59, 1024,   11,  513, 1024, 1024, 1024,  241,  241, 1024, 1024,    7,
        1024,  736, 1024, 1024,   63,   51,   31, 1024,   11,  332, 1024,  178,
          38, 1024,   58, 1024, 1024]), text='add two tablespoons thick sour cream two tablespoons sugar a sprinkle of mustard and half cup of vinegar', dec_out=None, dec_state=None, timestamp=[], alignments=None, frame_confidence=None, token_confidence=None, word_confidence=No

In [12]:
# fine tuning

from omegaconf import OmegaConf

# manifests location
train_manifest = "/content/manifests/train_manifest.json"
val_manifest = "/content/manifests/val_manifest.json"

asr_model.setup_training_data(
    train_data_config={
        "manifest_filepath": train_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": True,
    }
)

asr_model.setup_validation_data(
    val_data_config={
        "manifest_filepath": val_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": False,
    }
)

from omegaconf import open_dict

with open_dict(asr_model.cfg.optim):
    asr_model.cfg.optim.lr = 1e-4

[NeMo I 2026-05-28 10:20:28 nemo_logging:393] Dataset loaded with 2162 files totalling 4.28 hours
[NeMo I 2026-05-28 10:20:28 nemo_logging:393] 0 files were filtered totalling 0.00 hours
[NeMo I 2026-05-28 10:20:28 nemo_logging:393] Dataset loaded with 270 files totalling 0.55 hours
[NeMo I 2026-05-28 10:20:28 nemo_logging:393] 0 files were filtered totalling 0.00 hours


In [22]:
# training
# pytorch dataloaders and defining batch size and shuffle behaviour

import lightning.pytorch as pl

trainer = pl.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    log_every_n_steps=5
)

asr_model.set_trainer(trainer)
trainer.fit(asr_model)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[NeMo I 2026-05-28 10:43:44 nemo_logging:393] Optimizer config = Novograd (
    Parameter Group 0
        amsgrad: False
        betas: [0.8, 0.25]
        eps: 1e-08
        grad_averaging: False
        lr: 0.0001
        weight_decay: 0.001
    )
[NeMo I 2026-05-28 10:43:44 nemo_logging:393] Scheduler "<nemo.core.optim.lr_scheduler.CosineAnnealing object at 0x7b8dd972b8c0>" 
    will be used during training (effective maximum steps = 271) - 
    Parameters : 
    (warmup_steps: 1000
    warmup_ratio: null
    min_lr: 1.0e-05
    last_epoch: -1
    max_steps: 271
    )


INFO: 
  | Name              | Type                              | Params | Mode 
--------------------------------------------------------------------------------
0 | preprocessor      | AudioToMelSpectrogramPreprocessor | 0      | train
1 | encoder           | ConvASREncoder                    | 9.6 M  | train
2 | decoder           | ConvASRDecoder                    | 657 K  | train
3 | loss              | CTCLoss                           | 0      | train
4 | spec_augmentation | SpectrogramAugmentation           | 0      | train
5 | wer               | WER                               | 0      | train
--------------------------------------------------------------------------------
10.3 M    Trainable params
0         Non-trainable params
10.3 M    Total params
41.039    Total estimated model params size (MB)
943       Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name              | Type                              | Param

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


In [23]:
# evaluate WER on the test set
test_manifest = "/content/manifests/test_manifest.json"

asr_model.setup_test_data(
    test_data_config={
        "manifest_filepath": test_manifest,
        "sample_rate": 16000,
        "batch_size": 8,
        "shuffle": False,
    }
)

trainer.test(asr_model)

[NeMo I 2026-05-28 10:45:33 nemo_logging:393] Dataset loaded with 271 files totalling 0.55 hours
[NeMo I 2026-05-28 10:45:33 nemo_logging:393] 0 files were filtered totalling 0.00 hours


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        global_step        │           271.0           │
│         test_loss         │    5.5638813972473145     │
│         test_wer          │    0.04931942746043205    │
└───────────────────────────┴───────────────────────────┘

[{'global_step': 271.0,
  'test_loss': 5.5638813972473145,
  'test_wer': 0.04931942746043205}]

In [24]:
# save model
asr_model.save_to("/content/citrinet_librispeech_finetuned.nemo")

In [25]:
import json

test_items = []
with open("/content/manifests/test_manifest.json", "r", encoding="utf-8") as f:
    for line in f:
        test_items.append(json.loads(line))

subset = test_items[:20]
audio_files = [x["audio_filepath"] for x in subset]
references = [x["text"] for x in subset]

predictions = asr_model.transcribe(audio_files)

with open("/content/asr_test_predictions.txt", "w", encoding="utf-8") as f:
    for i, (ref, pred) in enumerate(zip(references, predictions), 1):
        pred_text = pred.text if hasattr(pred, "text") else str(pred)
        f.write(f"Example {i}\n")
        f.write(f"REF:  {ref}\n")
        f.write(f"PRED: {pred_text}\n")
        f.write("-" * 80 + "\n")

print("Saved to /content/asr_test_predictions.txt")

Transcribing: 100%|██████████| 5/5 [00:07<00:00,  1.41s/it]

Saved to /content/asr_test_predictions.txt


In [26]:
!git clone https://github.com/diaconescualexandra/ASR_TTS_NKUA.git

Cloning into 'ASR_TTS_NKUA'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 62 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 21.29 KiB | 5.32 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [28]:
%cd ASR_TTS_NKUA
!mkdir -p asr/checkpoints


/content/ASR_TTS_NKUA


In [29]:
!cp /content/citrinet_librispeech_finetuned.nemo asr/checkpoints/
!cp /content/asr_test_predictions.txt asr/evaluation/


In [30]:
!cp /content/manifests/*.json preprocessing/

In [31]:
!find asr -type f
!find preprocessing -type f

asr/evaluation/.gitkeep
asr/evaluation/asr_test_predictions.txt
asr/.gitkeep
asr/configs/.gitkeep
asr/checkpoints/citrinet_librispeech_finetuned.nemo
asr/training_code/.gitkeep
preprocessing/test_manifest.json
preprocessing/.gitkeep
preprocessing/train_manifest.json
preprocessing/val_manifest.json


In [32]:
!git add .
!git commit -m "Completed ASR fine-tuning and evaluation"
!git push

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@237512f9517e.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
